<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/preprocessing/training_codes_overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install xlsxwriter
!pip install pandas openpyxl

In [33]:
import json
labelset = []
with open('labelset.txt', 'r') as file:
    for line in file:
        labelset.append(line.strip())

data = []
with open('train_dataset.jsonl', 'r', encoding = 'utf-8') as f:
    for line in f:
        data.append(json.loads(line))

In [34]:
import pandas as pd

labelset = pd.DataFrame(labelset)
labelset.columns = ['codes']
labelset[['category', 'code']] = labelset['codes'].str.extract(r'([A-Z])([0-9]*)')

print(f"The labelset contains {len(labelset)} unique ICD-10 codes.")

The labelset contains 324 unique ICD-10 codes.


In [35]:
count_df = labelset.groupby('category').size().reset_index(name='count')

In [36]:
import pandas as pd
result_df = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']

    for annotation in entry['annotations']:
        result_df.append({
            'patient_id': patient_id,
            'start': annotation['start'],
            'end': annotation['end'],
            'code': annotation['code'],
            'mention': annotation['mention']
        })
result_df = pd.DataFrame(result_df)

In [37]:
result_df[['category', 'subcode']] = result_df['code'].str.extract(r'([A-Z])([0-9]*)')
unique_codes = result_df['code'].unique()
unique_codes_count = result_df['code'].drop_duplicates().shape[0]

print(f"The training set contains {unique_codes_count} unique ICD-10 codes.")

The training set contains 144 unique ICD-10 codes.


In [38]:
count_df = result_df.groupby('category').size().reset_index(name='count')
print(count_df)

   category  count
0         A      5
1         C     40
2         D     65
3         E    616
4         F      1
5         G      7
6         I   5142
7         J    454
8         K     14
9         M     25
10        N    209
11        Q      6
12        R   1895
13        T      7
14        Y    412
15        Z   1270


In [39]:
count_df = result_df.groupby('code').size().reset_index(name='count')
code_counts = count_df.sort_values(by='count', ascending = False).reset_index(drop=True)

df_above_500 = code_counts[code_counts['count'] > 500]
df_200_to_500 = code_counts[(code_counts['count'] >= 200) & (code_counts['count'] <= 500)]
df_20_to_200 = code_counts[(code_counts['count'] >= 20) & (code_counts['count'] <= 200)]
df_below_20 = code_counts[code_counts['count'] < 20]

code_list = df_20_to_200['code'].tolist()
code_df = pd.DataFrame(code_list, columns=["code"])
code_df.to_csv('codes_list.csv', index=False)
with open('codes_list.txt', 'w') as f:
    for code in code_list:
        f.write(f"{code}\n")

In [40]:
dfs = {
    'Above 500': df_above_500,
    '100 to 500': df_200_to_500,
    '100 to 50': df_20_to_200,
    'Below 100': df_below_20
}
file_path = 'code_frequencies.xlsx'
with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
    for sheet_name, df in dfs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [46]:
import pandas as pd

icd_10 = pd.read_excel("ICD-10_greek.xls", header=None, names=["codes", "description"])
icd_10['stripped_code'] = icd_10['codes'].astype(str).str.split('.').str[0]
grouped_icd_10 = icd_10.groupby('stripped_code')['description'].agg(list).reset_index()
labelset_with_descriptions = pd.merge(labelset, grouped_icd_10, left_on='codes', right_on='stripped_code', how='left')
labelset_with_descriptions = labelset_with_descriptions[['codes', 'description']]

#https://www.galinos.gr/web/drugs/main/icdcodes/C86
c86_descriptions = [
    "Εξωλεμφαδενικό Τ/ΝΚ κυτταρικό λέμφωμα, ρινικού τύπου",
    "Ηπατοσπληνικό λέμφωμα Τ κυττάρων",
    "Λέμφωμα Τ κυττάρων τύπου εντεροπάθειας",
    "Υποδόριο Τ-λέμφωμα μιμούμενο υποδερματίτιδα",
    "Βλαστικό λέμφωμα ΝΚ κυττάρων",
    "Αγγειοανοσοβλαστικό Τ- λέμφωμα",
    "Πρωτοπαθής CD30+ (θετική) Τ-λεμφοϋπερπλαστική αλλοίωση του δέρματος"
]

idx = labelset_with_descriptions[labelset_with_descriptions['codes'] == 'C86'].index[0]
labelset_with_descriptions.at[idx, 'description'] = c86_descriptions
desc_dict = labelset_with_descriptions.set_index('codes')['description'].to_dict()

with open("labelset_descriptions.json", "w", encoding="utf-8") as f:
    json.dump(desc_dict, f, ensure_ascii=False, indent=2)